In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / 'scripts'))

from tts_util import preparar_entorno, sin_ruido, cargar_frases, resumen_corpus
from tts_util import BASE, AUDIO, RES, MOS, EQUIPO

preparar_entorno()

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import Audio, display

import adaptadores, analisis, evaluacion

---
# Sesión 1 — Corpus y candidatos

## Tarea 1.1 — Corpus de frases

Todas las medidas de la práctica se hacen sobre las mismas 20 frases.
Partid de `frases_ejemplo.tsv`, copiadlo como `frases.tsv` y completadlo
con frases de **vuestro dominio**.

Composición obligatoria: 6 `decl`, 4 `int_abs`, 4 `int_parc`,
3 `cifras`, 3 `siglas`.

La columna `leida` es cómo **debería** pronunciarse la frase, en letras y
sin puntuación. Es la referencia para medir la inteligibilidad en la S3.

In [ ]:
# TAREA 1.1 — ejecutad esto una vez para crear vuestro frases.tsv,
# después editadlo desde el panel de ficheros de JupyterLab.
import shutil
destino = BASE / 'frases.tsv'
if not destino.exists():
    shutil.copy('frases_ejemplo.tsv', destino)
    print(f'Creado {destino}. Editadlo y volved aquí.')
else:
    print(f'{destino} ya existe.')

In [ ]:
frases = cargar_frases()
resumen_corpus(frases)

## Tarea 1.2 — Candidatos

Al menos **tres sistemas** que funcionen, de **al menos dos familias**
distintas. Piper viene preparado; el segundo es un ejemplo que puede o no
funcionar; el tercero lo buscáis vosotros.

Si un candidato no arranca: anotad el error exacto, dedicadle como mucho
30 minutos y descartadlo. Un sistema descartado con su diagnóstico cuenta
como candidato explorado en el informe.

In [ ]:
# PRUEBA
import os
from pathlib import Path

espeak_data = (
    Path.cwd()
    / ".venv"
    / "Lib"
    / "site-packages"
    / "piper"
    / "espeak-ng-data"
)

print("Ruta:", espeak_data)
print("Existe:", espeak_data.exists())
print("Contiene phondata:", (espeak_data / "phondata").exists())

os.environ["ESPEAK_DATA_PATH"] = str(espeak_data)
print("ESPEAK_DATA_PATH configurada")

In [ ]:
CANDIDATOS = {}

# Candidato A — Piper (CPU). Descarga la voz la primera vez (~74 MB).
adaptadores.registrar(CANDIDATOS, 'piper_sharvard',
                      adaptadores.crear_piper, 'es_ES-sharvard-medium')

# Candidato B — MMS-TTS (GPU). Si falla, anotad el error y seguid.
adaptadores.registrar(CANDIDATOS, 'mms_spa',
                      adaptadores.crear_mms)

# TAREA 1.2 — Candidato C: buscadlo vosotros y registradlo aquí.
# Candidato C - Bark (Transformers / Generativo)
adaptadores.registrar(CANDIDATOS, 'bark_spa',
                      adaptadores.crear_bark)

print()
print('Candidatos activos:', list(CANDIDATOS))

In [ ]:
# Prueba rápida: una frase con cada candidato
for nombre, tts in CANDIDATOS.items():
    ruta = AUDIO / 'prueba' / f'{nombre}.wav'
    tts(frases[0]['texto'], ruta)
    print(nombre)
    display(Audio(str(ruta)))

## Tarea 1.3 — Ficha de cada candidato

Rellenad esta tabla en el informe (aquí podéis ir tomando notas).
Las tres primeras columnas se completan hoy; la calidad, en la S3.

| Criterio | Dónde se mira |
|---|---|
| Licencia y uso comercial | *model card*, fichero LICENSE del repositorio |
| Español: variedad, nº de voces | *model card* y vuestra escucha |
| Memoria ocupada | `torch.cuda.memory_allocated()` o `nvidia-smi` |
| Velocidad (RTF y carga) | tarea 1.4 |
| Mantenimiento | última versión, actividad, dependencias |
| Calidad | sesión 3 |

## Tarea 1.4 — Velocidad

`medir_rtf` separa la carga del modelo del tiempo de síntesis, descarta
una primera llamada de calentamiento y toma la mediana de varias
repeticiones. Las tres cosas hay que saber justificarlas en la defensa.

In [ ]:
filas = []
for nombre, tts in CANDIDATOS.items():
    print(f'Midiendo {nombre} ...')
    for f in analisis.medir_rtf(tts, frases, AUDIO / nombre):
        filas.append(dict(sistema=nombre, **f))

rtf = pd.DataFrame(filas)
rtf.to_csv(RES / 'rtf.csv', index=False)
rtf.groupby('sistema')[['rtf', 'dur']].describe().round(3)

In [ ]:
# Tiempo de CARGA de cada modelo, que se reporta aparte del RTF
_, t_piper = analisis.medir_carga(adaptadores.crear_piper, 'es_ES-sharvard-medium')
print(f'Carga de Piper: {t_piper} s')

# TAREA 1.4 — Repetid la medida de Piper con usar_gpu=True y comparad.
# ¿Gana siempre la GPU con un modelo pequeño? Justificadlo.

---
# Sesión 2 — Control prosódico

La pregunta de esta sesión no es si suena distinto, sino **qué cambia y
qué no** al tocar los parámetros o el texto.

`analisis.analiza(ruta)` devuelve la duración, la F0 mediana, el recorrido
de F0 en semitonos y `final_st`: el movimiento de los últimos 300 ms.
Positivo = termina subiendo; negativo = termina bajando.

## Tarea 2.1 — ¿Es lineal el control de velocidad?

In [ ]:
# PRUEBA - COMPROBAR QUE CANDIDATOS ADMITEN PARAMETROS
for nombre, tts in CANDIDATOS.items():
    print(nombre, "->", getattr(tts, "admite_parametros", False))

In [ ]:
escalas = [0.6, 0.8, 1.0, 1.2, 1.4, 1.6]
decl = [f for f in frases if f['categoria'] == 'decl']

tts = CANDIDATOS['piper_sharvard']   # solo sistemas con admite_parametros
filas = []
for fr in decl:
    for e in escalas:
        ruta = AUDIO / 'velocidad' / f"{fr['id']}_{e}.wav"
        tts(fr['texto'], ruta, length_scale=e)
        filas.append(dict(id=fr['id'], escala=e, dur=analisis.analiza(ruta)['dur']))

vel = pd.DataFrame(filas)
base = vel[vel.escala == 1.0].set_index('id').dur
vel['dur_rel'] = vel.dur / vel.id.map(base)
vel.groupby('escala').dur_rel.mean().round(3)

In [ ]:
# TAREA 2.1 — Dibujad dur_rel frente a escala junto con la diagonal y = x.
import matplotlib.pyplot as plt

medias = vel.groupby('escala')['dur_rel'].mean()

plt.figure(figsize=(7, 5))
plt.plot(medias.index, medias.values, 'o-', label='Duración relativa medida')
plt.plot([0.6, 1.6], [0.6, 1.6], '--', label='Referencia y = x')
plt.xlabel('length_scale')
plt.ylabel('Duración relativa')
plt.title('Relación entre length_scale y duración')
plt.grid(True)
plt.legend()
plt.show()

# ¿Se ajusta? ¿Se alargan igual los silencios que los sonidos?

# La duración total de los audios aumenta al incrementar `length_scale`, aunque no de forma exactamente proporcional. Al escuchar los audios, parece que los sonidos se alargan más que los silencios, por lo que ambos no se modifican necesariamente en la misma proporción.

# ¿A partir de qué valor la voz deja de ser aceptable? (escuchadlo)

# Los valores más aceptables parecen ser 1.0 y 1.2. Para valores más bajos, la voz se escucha demasiado rápida y algunas palabras pueden resultar difíciles de distinguir. Para valores más altos, el habla resulta excesivamente lenta y puede sonar artificial.


## Tarea 2.2 — ¿Es determinista el sistema?

In [ ]:
# TAREA 2.2 — Sintetizad la MISMA frase 5 veces en tres condiciones:
#   a) parámetros por defecto
#   b) noise_scale=0.0, noise_w_scale=0.0
#   c) noise_scale=1.0, noise_w_scale=1.0

tts = CANDIDATOS['piper_sharvard']

# Misma frase para las tres condiciones
fr = decl[0]

condiciones = {
    'defecto': {},
    'ruido_0': {
        'noise_scale': 0.0,
        'noise_w_scale': 0.0
    },
    'ruido_1': {
        'noise_scale': 1.0,
        'noise_w_scale': 1.0
    }
}

repeticiones = 5
filas = []


for nombre_cond, parametros in condiciones.items():
    for i in range(repeticiones):
        ruta = AUDIO / 'variabilidad' / f'{nombre_cond}_{i}.wav'
        ruta.parent.mkdir(parents=True, exist_ok=True)

        tts(fr['texto'], ruta, **parametros)

        medidas = analisis.analiza(ruta)

        filas.append({
            'condicion': nombre_cond,
            'repeticion': i + 1,
            'dur': medidas['dur'],
            'rango_st': medidas['rango_st']
        })

variabilidad = pd.DataFrame(filas)
variabilidad



In [ ]:
# Para cada grupo, desviación típica de la duración y del rango_st.
resumen_22 = variabilidad.groupby('condicion').agg(
    dur_media=('dur', 'mean'),
    dur_std=('dur', 'std'),
    rango_st_medio=('rango_st', 'mean'),
    rango_st_std=('rango_st', 'std')
).round(4)

resumen_22

Con los parámetros por defecto se observa cierta variabilidad entre las cinco síntesis, con una desviación típica de 0,0621 s en la duración y de 0,9739 en el rango ST.

Al establecer `noise_scale=0.0` y `noise_w_scale=0.0`, ambas desviaciones típicas pasan a ser cero. Por tanto, las cinco síntesis presentan los mismos valores de duración y rango ST, lo que indica que se ha eliminado la variabilidad observada en estas medidas.

Con ambos parámetros a 1.0, vuelve a existir variabilidad: la desviación típica de la duración es de 0,0406 s y la del rango ST es de 1,7023. En esta condición, la variabilidad del rango ST es mayor que con los parámetros por defecto.

Para determinar si los audios con variabilidad cero suenan más monótonos, es necesario comparar auditivamente las grabaciones de las tres condiciones.

### ¿Con variabilidad cero salen audios idénticos? ¿Suena más monótona?

Al escuchar los audios, parece que las cinco repeticiones con variabilidad cero son idénticas. En las condiciones por defecto y con los parámetros de variabilidad a 1.0, la entonación cambia entre algunas repeticiones. En cambio, con variabilidad cero, los audios suenan más monótonos y mantienen la misma entonación.

## Tarea 2.3 — Modalidad: ¿pregunta cuando debe?

Ningún sistema tiene un parámetro «pregunta»: la entonación se infiere de
la puntuación. Comprobad si lo hace bien.

In [ ]:
fr = decl[0]['texto'].rstrip('.')
versiones = {'.': fr + '.', '?': '¿' + fr + '?', '!': '¡' + fr + '!'}

etiquetadas = []
for nombre, tts in CANDIDATOS.items():
    for marca, texto in versiones.items():

        #ruta = AUDIO / 'modalidad' / f'{nombre}_{marca.strip(chr(46))or "pt"}.wav'
        etiquetas = {'.': 'pt', '?': 'int', '¿': 'int', ',': 'coma'}
        ruta = AUDIO / 'modalidad' / f'{nombre}_{etiquetas.get(marca, "otro")}.wav'

        tts(texto, ruta)
        a = analisis.analiza(ruta)
        print(f'{nombre:20s} {marca}  final_st = {a["final_st"]:+.2f}')
        if nombre == 'piper_sharvard':
            etiquetadas.append((marca, ruta))

analisis.dibuja_contornos(etiquetadas, 'Modalidad — Piper',
                          RES / 'modalidad_piper.png');

In [ ]:

fr = decl[0]['texto'].rstrip(' .')

versiones = {
    '.': fr + '.',
    '?': '¿' + fr + '?',
    '!': '¡' + fr + '!'
}

# Guardar los contornos por sistema
etiquetadas_por_sistema = {
    nombre: [] for nombre in CANDIDATOS
}

# Etiquetas seguras para nombres de archivo en Windows
etiquetas_archivo = {
    '.': 'pt',
    '?': 'int',
    '!': 'exc'
}

for nombre, tts in CANDIDATOS.items():
    for marca, texto in versiones.items():
        ruta = AUDIO / 'modalidad' / (
            f'{nombre}_{etiquetas_archivo[marca]}.wav'
        )

        tts(texto, ruta)
        a = analisis.analiza(ruta)

        print(
            f'{nombre:20s} {marca}  '
            f'final_st = {a["final_st"]:+.2f} st'
        )

        etiquetadas_por_sistema[nombre].append((marca, ruta))

# Dibujar los tres contornos de cada sistema
for nombre, etiquetadas in etiquetadas_por_sistema.items():
    titulo = f'Modalidad — {nombre}'
    salida = RES / f'modalidad_{nombre}.png'

    analisis.dibuja_contornos(etiquetadas, titulo, salida)


In [ ]:
# TAREA 2.3 — Aplicad lo mismo a vuestras 4 int_abs y 4 int_parc.
# Seleccionar las 4 frases de cada categoría
interrogativas = {
    'int_abs': [f for f in frases if f['categoria'] == 'int_abs'][:4],
    'int_parc': [f for f in frases if f['categoria'] == 'int_parc'][:4]
}

filas_int = []
etiquetas_archivo = {
    'int_abs': 'abs',
    'int_parc': 'parc'
}

for nombre, tts in CANDIDATOS.items():
    etiquetadas = []

    for categoria, lista in interrogativas.items():
        for fr in lista:
            texto = fr['texto'].strip()
            ruta = AUDIO / 'interrogativas' / (
                f'{nombre}_{etiquetas_archivo[categoria]}_{fr["id"]}.wav'
            )

            # Sintetizar y analizar
            tts(texto, ruta)
            a = analisis.analiza(ruta)

            filas_int.append({
                'sistema': nombre,
                'categoria': categoria,
                'id': fr['id'],
                'texto': texto,
                'dur': a['dur'],
                'final_st': a['final_st']
            })

            # Etiqueta para distinguir cada frase en la gráfica
            etiquetadas.append((f'{categoria}_{fr["id"]}', ruta))

    # Dibujar los contornos de las 8 interrogativas del sistema
    analisis.dibuja_contornos(
        etiquetadas,
        f'Interrogativas — {nombre}',
        RES / f'interrogativas_{nombre}.png'
    )

# Tabla de resultados
interrogativas_resultados = pd.DataFrame(filas_int)

display(interrogativas_resultados)

# Resumen de final_st por sistema y categoría
resumen_int = interrogativas_resultados.groupby(
    ['sistema', 'categoria']
).agg(
    final_st_medio=('final_st', 'mean'),
    final_st_std=('final_st', 'std'),
    dur_media=('dur', 'mean')
).round(2)

display(resumen_int)

# Guardar los resultados
interrogativas_resultados.to_csv(
    RES / 'interrogativas_23.csv', index=False
)



### En español la interrogativa absoluta suele terminar en ascenso y la parcial no necesariamente: ¿lo reproduce cada sistema?

Al analizar las cuatro interrogativas absolutas y las cuatro parciales, se observa que ninguno de los tres sistemas reproduce de forma consistente un ascenso final en las interrogativas absolutas. Piper y MMS-TTS presentan mayoritariamente valores negativos de `final_st`, mientras que Bark muestra una variabilidad mayor, con algunos ascensos finales pronunciados y otros descensos.

Por tanto, los resultados sugieren que los sistemas no reproducen de manera uniforme la diferencia entonativa esperada entre interrogativas absolutas y parciales. Esta conclusión debe contrastarse con la escucha de los audios, ya que el valor `final_st` no representa por sí solo todo el contorno de entonación.

## Tarea 2.4 — Foco: ¿se puede enfatizar una palabra?

Los sistemas no tienen control de foco. Solo podéis intentarlo desde el
texto: mayúsculas, comas, reordenar, repetir, partir en dos oraciones.
Que no funcione nada es un resultado válido y hay que documentarlo.

In [ ]:
# TAREA 2.4 — Probad al menos tres trucos sobre una frase, poniendo el
# énfasis en tres palabras distintas. Medid F0 máxima y duración de la
# palabra objetivo en cada versión.

frase_base = "El plazo termina el viernes"

experimentos = {
    "plazo": {
        "normal": "El plazo termina el viernes.",
        "mayusculas": "El PLAZO termina el viernes.",
        "repeticion": "El plazo plazo termina el viernes.",
        "pausa": "El plazo... termina el viernes."
    },
    "termina": {
        "normal": "El plazo termina el viernes.",
        "mayusculas": "El plazo TERMINA el viernes.",
        "repeticion": "El plazo termina termina el viernes.",
        "pausa": "El plazo termina... el viernes."
    },
    "viernes": {
        "normal": "El plazo termina el viernes.",
        "mayusculas": "El plazo termina el VIERNES.",
        "repeticion": "El plazo termina el viernes viernes.",
        "pausa": "El plazo termina el viernes." # No hace falta poner pausa
    }
}

filas_foco = []

for nombre, tts in CANDIDATOS.items():
    for objetivo, versiones_obj in experimentos.items():
        for truco, texto in versiones_obj.items():
            ruta = AUDIO / "foco" / (
                f"{nombre}_{objetivo}_{truco}.wav"
            )

            tts(texto, ruta)
            a = analisis.analiza(ruta)

            filas_foco.append({
                "sistema": nombre,
                "objetivo": objetivo,
                "truco": truco,
                "texto": texto,
                "dur_total": a["dur"]
            })

foco = pd.DataFrame(filas_foco)
# display(foco)


In [ ]:
filas_foco = []

for nombre, tts in CANDIDATOS.items():
    for objetivo, versiones_obj in experimentos.items():
        for truco, texto in versiones_obj.items():
            ruta = AUDIO / "foco" / f"{nombre}_{objetivo}_{truco}.wav"

            tts(texto, ruta)
            a = analisis.analiza(ruta)

            filas_foco.append({
                "sistema": nombre,
                "objetivo": objetivo,
                "truco": truco,
                "texto": texto,
                "dur_total": a["dur"],
                "f0_max_audio": float(np.max(a["f0"])) if len(a["f0"]) else np.nan
            })

foco = pd.DataFrame(filas_foco)
display(foco)

## Tarea 2.5 — Normalización de texto

In [ ]:
# Escuchad las frases de las categorías cifras y siglas en los 3 sistemas
for fr in [f for f in frases if f['categoria'] in ('cifras', 'siglas')]:
    print(fr['texto'])
    for nombre in CANDIDATOS:
        display(Audio(str(AUDIO / nombre / f"{fr['id']}.wav")))

In [ ]:
# TAREA 2.5 — Escribid normaliza(texto) para convertir fechas, horas,
# importes y siglas a letras ANTES de sintetizar, y comparad el resultado.
import re

def normaliza(texto):
    # Fechas en formato DD/MM: por ejemplo, 30/09.
    meses = {
        1: "enero", 2: "febrero", 3: "marzo",
        4: "abril", 5: "mayo", 6: "junio",
        7: "julio", 8: "agosto", 9: "septiembre",
        10: "octubre", 11: "noviembre", 12: "diciembre"
    }

    def convertir_fecha(match):
        dia = int(match.group(1))
        mes = int(match.group(2))

        if 1 <= mes <= 12:
            return f"{numero_a_letras(dia)} de {meses[mes]}"

        return match.group(0)

    # Horas en formato HH:MM, por ejemplo 14:00 o 09:30.
    def convertir_hora(match):
        hora = int(match.group(1))
        minutos = int(match.group(2))

        if 0 <= hora <= 23 and 0 <= minutos <= 59:
            return hora_a_letras(hora, minutos)

        return match.group(0)

    # Números aislados que no se hayan convertido como fecha u hora.
    def convertir_numero(match):
        numero = int(match.group(0))
        return numero_a_letras(numero)

    texto = re.sub(
        r'\b(\d{1,2}):(\d{2})\b',
        convertir_hora,
        texto
    )


    texto = re.sub(
        r'\b(\d{1,2})/(\d{1,2})\b',
        convertir_fecha,
        texto
    )

    texto = re.sub(
        r'(?<![\w/:])\d{1,6}(?![\w/:])',
        convertir_numero,
        texto
    )

    # Ejemplos de siglas del dominio.
    siglas = {
        "ETSI": "e te ese i",
        "TFG": "te efe ge",
        "DNI": "de ene i",
        "UHU": "u ache u",
        "BOE": "be o e"
    }

    for sigla, pronunciacion in siglas.items():
        texto = re.sub(
            rf'\b{sigla}\b',
            pronunciacion,
            texto,
            flags=re.IGNORECASE
        )

    return texto




def numero_a_letras(n):
    if not isinstance(n, int) or isinstance(n, bool) or n < 0 or n > 999999:
        raise ValueError("El número debe estar entre 0 y 999999")

    unidades = [
        "cero", "uno", "dos", "tres", "cuatro",
        "cinco", "seis", "siete", "ocho", "nueve",
        "diez", "once", "doce", "trece", "catorce",
        "quince", "dieciséis", "diecisiete",
        "dieciocho", "diecinueve", "veinte",
        "veintiuno", "veintidós", "veintitrés",
        "veinticuatro", "veinticinco", "veintiséis",
        "veintisiete", "veintiocho", "veintinueve"
    ]

    decenas = {
        30: "treinta",
        40: "cuarenta",
        50: "cincuenta",
        60: "sesenta",
        70: "setenta",
        80: "ochenta",
        90: "noventa"
    }

    centenas = {
        100: "cien",
        200: "doscientos",
        300: "trescientos",
        400: "cuatrocientos",
        500: "quinientos",
        600: "seiscientos",
        700: "setecientos",
        800: "ochocientos",
        900: "novecientos"
    }

    def menos_de_mil(num):
        if num < 30:
            return unidades[num]

        if num < 100:
            decena = (num // 10) * 10
            resto = num % 10

            if resto == 0:
                return decenas[decena]

            return f"{decenas[decena]} y {unidades[resto]}"

        if num == 100:
            return "cien"

        if num < 200:
            return "ciento " + menos_de_mil(num - 100)

        centena = (num // 100) * 100
        resto = num % 100

        if resto == 0:
            return centenas[centena]

        return f"{centenas[centena]} {menos_de_mil(resto)}"

    if n < 1000:
        return menos_de_mil(n)

    miles = n // 1000
    resto = n % 1000

    if miles == 1:
        resultado = "mil"
    else:
        resultado = f"{menos_de_mil(miles)} mil"

    if resto:
        resultado += " " + menos_de_mil(resto)

    return resultado


    if n in unidades:
        return unidades[n]

    if 31 <= n <= 39:
        return "treinta y " + numero_a_letras(n - 30)

    if 41 <= n <= 49:
        return "cuarenta y " + numero_a_letras(n - 40)

    if 51 <= n <= 59:
        return "cincuenta y " + numero_a_letras(n - 50)

    raise ValueError("El número debe estar entre 0 y 59")

def hora_a_letras(hora, minutos):
    # Horas en punto: 14:00 -> catorce horas
    if minutos == 0:
        return f"{numero_a_letras(hora)} horas"

    # Horas con minutos: 14:30 -> catorce y treinta
    return (
        f"{numero_a_letras(hora)} horas y "
        f"{numero_a_letras(minutos)} minutos"
    )



Contempla numeros desde el 0 hasta el 999999

In [ ]:
# PROBAR FUNCIÓN
from IPython.display import Audio, display

frases_normalizacion = [
    fr for fr in frases
    if fr["categoria"] in ("cifras", "siglas")
]

for fr in frases_normalizacion:
    texto_norm = normaliza(fr["texto"])

    print(f"\n{fr['id']}: {fr['texto']}")
    print("Normalizado:", texto_norm)

    for nombre, tts in CANDIDATOS.items():
        ruta = AUDIO / "normalizadas" / nombre / f"{fr['id']}.wav"
        tts(texto_norm, ruta)

        print(f"Sistema: {nombre}")
        display(Audio(filename=str(ruta)))


---
# Sesión 3 — Evaluación de calidad

Tres vías que responden a preguntas distintas. Lo interesante aparece
cuando no coinciden.

## Tarea 3.1 — Inteligibilidad

Se transcribe la voz sintética con el reconocedor de referencia de la
asignatura (el mismo para todos los equipos) y se compara con la columna
`leida`. Ocupa VRAM: cargadlo cuando vayáis a usarlo.

In [ ]:
intel = evaluacion.mide_inteligibilidad(CANDIDATOS, frases)
intel.to_csv(RES / 'inteligibilidad.csv', index=False)
intel.pivot_table(index='categoria', columns='sistema', values='wer').round(3)

**Separad el habla normal de las cifras y siglas.** En esas dos categorías
buena parte del WER no es falta de inteligibilidad, sino desacuerdo entre
vuestra columna `leida` y cómo escribe el reconocedor.

In [ ]:
evaluacion.resumen_wer(intel)

Ahora revisad **a mano** las frases difíciles. Para cada una, decidid si el
error es artefacto de convención (la voz se entendió, se escribió distinto)
o fallo real de síntesis (la voz dijo algo que no era). Esa clasificación
va al informe.

In [ ]:
evaluacion.revisar(intel)

In [ ]:
# TAREA 3.1 — Revisad A MANO las transcripciones de la categoría cifras.
# ¿Cuántos "errores" son en realidad '30' frente a 'treinta'?
intel[intel.categoria == 'cifras'][['sistema', 'referencia', 'hipotesis', 'wer']]

## Tarea 3.2 — MOS

Panel cruzado y ciego: vuestras muestras las puntúan otros equipos.
Elegid 5 frases (al menos una de cada categoría).

In [ ]:
# TAREA 3.2 — poned aquí vuestros 5 ids
elegidas = ['f01', 'f03', 'f04', 'f05', 'f06']

clave = evaluacion.preparar_mos(CANDIDATOS, frases, elegidas)

In [ ]:
# Cuando recibáis las notas (CSV con columnas evaluador, muestra, nota):
# evaluacion.analiza_mos(RES / 'mos_recibidas.csv')

## Tarea 3.3 — Consistencia de la voz

In [ ]:
emb, intra, entre = evaluacion.consistencia(CANDIDATOS, frases)

In [ ]:
# TAREA 3.3 — Comparad la consistencia de las frases interrogativas frente
# a las declarativas. ¿Cambia la "identidad" de la voz cuando pregunta?

## Tarea 3.4 — Reflexión ética

500-800 palabras en el informe, con al menos dos fuentes citadas.
Apoyaos en lo que habéis oído esta semana: ¿podrían estas voces engañar a
alguien? ¿En qué se nota que son sintéticas?

---
## Decisión final

¿Qué sistema usaríais en el asistente de vuestro dominio y por qué?
Ponderad calidad, velocidad, recursos y licencia. «El de mayor MOS» no
basta si es diez veces más lento o su licencia no lo permite.

In [ ]:
# Resumen de todas las medidas, para la discusión del informe
resumen = rtf.groupby('sistema').rtf.median().rename('rtf_mediano').to_frame()
resumen['wer_intel'] = intel.groupby('sistema').wer.mean()
resumen['consistencia'] = pd.Series(intra)
resumen.round(3)

# GUARDAR VARIABLES

In [ ]:
vel.to_csv(RES / 'velocidad_resultados.csv', index=False)

# CARGAR VARIABLES

In [ ]:
vel = pd.read_csv(RES / 'velocidad_resultados.csv')